# Día 9 — Ejercicios: fuentes de datos y estadística aplicada

Seis ejercicios de dificultad creciente que combinan la mañana (SQLModel, APIs) y la tarde (SciPy: distribuciones, pruebas de hipótesis) del día. Cada ejercicio indica su **criterio de validación explícito** — cómo saber, sin mirar la solución, si vuestra respuesta es correcta.

Cada ejercicio incluye una función `verificar_ejercicio_N()` **definida pero no ejecutada**. Completa primero la celda con `# Tu código aquí`; luego descomenta la línea que llama a `verificar_ejercicio_N()` para comprobar tu solución.

No abráis `08_soluciones.ipynb` antes de intentarlo por vuestra cuenta.


## Preparación común

Ejecuta esta celda antes de empezar: carga los datasets que usaremos en varios ejercicios.


In [ ]:
import json
import os
import tempfile
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from typing import Optional
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
import scipy.stats
from sqlalchemy.pool import StaticPool
from sqlmodel import Field, Session, SQLModel, and_, create_engine, or_, select

cartera = pd.read_csv("../data/raw/cartera_polizas.csv")
siniestros = pd.read_csv("../data/raw/siniestros.csv")

ALFA = 0.05
print(f"{len(cartera)} pólizas, {len(siniestros)} siniestros cargados.")


---
## Ejercicio 1 (fácil) — Create + Read con SQLModel

Define un modelo `Cliente` (`table=True`) con: `id` (PK autogenerada, opcional), `cliente_id: str`, `nombre: str`, `provincia: str`, `edad: int`. Crea un `engine` SQLite en memoria correctamente configurado para soportar varias sesiones (recordad `poolclass=StaticPool`), crea las tablas, e inserta estos 5 clientes:

```python
clientes_datos = [
    {"cliente_id": "C1", "nombre": "Anna Puig", "provincia": "barcelona", "edad": 42},
    {"cliente_id": "C2", "nombre": "Marc Soler", "provincia": "tarragona", "edad": 35},
    {"cliente_id": "C3", "nombre": "Laura Ruiz", "provincia": "madrid", "edad": 58},
    {"cliente_id": "C4", "nombre": "Jordi Camps", "provincia": "girona", "edad": 29},
    {"cliente_id": "C5", "nombre": "Marta Vidal", "provincia": "barcelona", "edad": 61},
]
```

Después, consulta (con `select`/`.where()`) los clientes con `edad >= 40`, y guarda el resultado en una variable `clientes_mayores_40` (lista de instancias `Cliente`).

**Criterio de validación**: `len(clientes_mayores_40)` debe ser igual a `sum(1 for c in clientes_datos if c["edad"] >= 40)` calculado directamente sobre la lista de diccionarios original (sin pasar por la base de datos) — si no coincide, algo falló en la inserción o en el filtro.


In [ ]:
clientes_datos = [
    {"cliente_id": "C1", "nombre": "Anna Puig", "provincia": "barcelona", "edad": 42},
    {"cliente_id": "C2", "nombre": "Marc Soler", "provincia": "tarragona", "edad": 35},
    {"cliente_id": "C3", "nombre": "Laura Ruiz", "provincia": "madrid", "edad": 58},
    {"cliente_id": "C4", "nombre": "Jordi Camps", "provincia": "girona", "edad": 29},
    {"cliente_id": "C5", "nombre": "Marta Vidal", "provincia": "barcelona", "edad": 61},
]

# Tu código aquí: modelo Cliente, engine, create_all, insertar clientes_datos,
# consultar edad >= 40 y guardar el resultado en clientes_mayores_40
pass


In [ ]:
def verificar_ejercicio_1():
    esperado = sum(1 for c in clientes_datos if c["edad"] >= 40)
    print(f"Esperado: {esperado} | Obtenido: {len(clientes_mayores_40)}")
    assert len(clientes_mayores_40) == esperado, "El número de clientes con edad >= 40 no coincide"
    print("Ejercicio 1 OK")


# Cuando hayas completado la celda anterior, descomenta la siguiente línea:
# verificar_ejercicio_1()


---
## Ejercicio 2 (fácil-medio) — Consumir una API local con `requests`

Se te da ya montado un servidor HTTP local (ejecuta la celda siguiente) que expone `GET /clientes` (todos) y `GET /clientes/<cliente_id>` (uno), protegido con un token `Bearer`. El token correcto es `"ejercicio-token-002"`.

Completa la función `consultar_cliente(cliente_id, token)` para que:
1. Haga la petición GET al endpoint correcto con la cabecera `Authorization` adecuada.
2. Si la respuesta es `200`, devuelva el JSON parseado (un `dict`).
3. Si es `401`, lance `PermissionError`.
4. Si es `404`, devuelva `None` (sin lanzar excepción).

**Criterio de validación**: `consultar_cliente("C1", "ejercicio-token-002")` debe devolver un dict con `"nombre": "Anna Puig"`; `consultar_cliente("C1", "token-malo")` debe lanzar `PermissionError`; `consultar_cliente("C999", "ejercicio-token-002")` debe devolver `None`.


In [ ]:
def crear_handler_ejercicio(token_esperado: str, datos: dict):
    class Handler(BaseHTTPRequestHandler):
        def _responder(self, status, payload):
            cuerpo = json.dumps(payload, ensure_ascii=False).encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.end_headers()
            self.wfile.write(cuerpo)

        def do_GET(self):
            if self.headers.get("Authorization") != f"Bearer {token_esperado}":
                self._responder(401, {"error": "no autorizado"})
                return
            ruta = urlparse(self.path).path
            if ruta == "/clientes":
                self._responder(200, {"clientes": list(datos.values())})
            elif ruta.startswith("/clientes/"):
                cid = ruta.removeprefix("/clientes/")
                if cid in datos:
                    self._responder(200, datos[cid])
                else:
                    self._responder(404, {"error": "no encontrado"})
            else:
                self._responder(404, {"error": "ruta no reconocida"})

        def log_message(self, format, *args):
            pass

    return Handler


datos_clientes_api = {c["cliente_id"]: c for c in clientes_datos}
servidor_ej2 = HTTPServer(("localhost", 0), crear_handler_ejercicio("ejercicio-token-002", datos_clientes_api))
puerto_ej2 = servidor_ej2.server_address[1]
threading.Thread(target=servidor_ej2.serve_forever, daemon=True).start()
base_url_ej2 = f"http://localhost:{puerto_ej2}"
print(f"Servidor del ejercicio 2 escuchando en {base_url_ej2}")


In [ ]:
def consultar_cliente(cliente_id: str, token: str) -> Optional[dict]:
    # Tu código aquí
    pass


In [ ]:
def verificar_ejercicio_2():
    resultado_ok = consultar_cliente("C1", "ejercicio-token-002")
    print("C1 con token correcto:", resultado_ok)
    assert resultado_ok is not None and resultado_ok["nombre"] == "Anna Puig"

    try:
        consultar_cliente("C1", "token-malo")
        raise AssertionError("Debería haber lanzado PermissionError")
    except PermissionError:
        print("Token incorrecto -> PermissionError (correcto)")

    resultado_404 = consultar_cliente("C999", "ejercicio-token-002")
    print("Cliente inexistente:", resultado_404)
    assert resultado_404 is None
    print("Ejercicio 2 OK")


# Cuando hayas completado consultar_cliente, descomenta la siguiente línea:
# verificar_ejercicio_2()


No olvides cerrar el servidor de prueba de este ejercicio cuando termines (idempotencia, igual que en el notebook de teoría):


In [ ]:
servidor_ej2.shutdown()
servidor_ej2.server_close()
print("Servidor del ejercicio 2 cerrado.")


---
## Ejercicio 3 (medio) — Update y Delete

Usando la base de datos del Ejercicio 1 (`engine`, tabla `Cliente`): actualiza la edad de `"C2"` a `40`, y elimina el cliente `"C4"`.

**Criterio de validación**: tras los cambios, una consulta `select(Cliente)` completa debe devolver 4 clientes (uno menos que al principio); el cliente `"C2"` debe tener `edad == 40`; el cliente `"C4"` no debe aparecer en ningún resultado.


In [ ]:
# Tu código aquí: UPDATE de C2 (edad=40) y DELETE de C4, cada uno con su propio commit
pass


In [ ]:
def verificar_ejercicio_3():
    with Session(engine) as session:
        todos = session.exec(select(Cliente)).all()
        c2 = session.exec(select(Cliente).where(Cliente.cliente_id == "C2")).first()
        c4 = session.exec(select(Cliente).where(Cliente.cliente_id == "C4")).first()

    print(f"Total de clientes: {len(todos)}")
    assert len(todos) == 4, "Debería quedar 1 cliente menos tras el DELETE"
    assert c2 is not None and c2.edad == 40, "C2 debería tener edad=40"
    assert c4 is None, "C4 debería haber sido eliminado"
    print("Ejercicio 3 OK")


# Requiere haber completado el Ejercicio 1 antes (usa el mismo engine y modelo Cliente).
# Cuando hayas completado la celda anterior, descomenta la siguiente línea:
# verificar_ejercicio_3()


---
## Ejercicio 4 (medio) — Estadística descriptiva aplicada

Calcula, sobre `siniestros`, la severidad media **ponderada** por número de siniestros de cada producto (equivalente a la media directa de `importe_pagado` sobre todo el DataFrame) y compárala con la media simple de las medias por producto (que trata cada producto por igual, sin tener en cuenta cuántos siniestros tiene cada uno).

Guarda los dos resultados en `media_ponderada` y `media_simple_de_medias`.

**Criterio de validación**: `media_ponderada` debe coincidir (con una tolerancia de 0.01 €) con `siniestros["importe_pagado"].mean()` calculado directamente sobre todo el DataFrame.


In [ ]:
# Tu código aquí
pass


In [ ]:
def verificar_ejercicio_4():
    media_directa = siniestros["importe_pagado"].mean()
    print(f"Media ponderada calculada: {media_ponderada:.4f}")
    print(f"Media directa (control):   {media_directa:.4f}")
    assert abs(media_ponderada - media_directa) < 0.01, "La media ponderada debería coincidir con la media directa"
    print("Ejercicio 4 OK")
    print(f"\nPara comparar: media simple de las medias por producto = {media_simple_de_medias:.4f}")
    print("(normalmente distinta de la media ponderada -- por eso importa elegir la correcta)")


# Cuando hayas completado la celda anterior, descomenta la siguiente línea:
# verificar_ejercicio_4()


---
## Ejercicio 5 (medio-difícil) — Distribuciones de probabilidad

Sobre la cartera, calcula la proporción histórica de pólizas de `HOGAR` que están `anulada == True` (llámala `p_anulada_hogar`). Usando una distribución **binomial** con `n=15` (pólizas nuevas de HOGAR) y esa `p`, calcula la probabilidad de que **como mucho 1** de esas 15 pólizas se anule en su primer año (`prob_maximo_1`).

Después, valida numéricamente comparando la probabilidad teórica con una simulación: genera 5.000 muestras de 15 ensayos con `scipy.stats.binom(...).rvs(size=5000, random_state=42)`, calcula la proporción de esas 5.000 simulaciones en las que hubo como mucho 1 anulación, y guárdala en `prob_maximo_1_simulada`.

**Criterio de validación**: `abs(prob_maximo_1 - prob_maximo_1_simulada)` debe ser menor que `0.03` (una simulación de 5.000 repeticiones debería acercarse bastante al valor teórico exacto).


In [ ]:
# Tu código aquí
pass


In [ ]:
def verificar_ejercicio_5():
    print(f"Probabilidad teórica (cdf):     {prob_maximo_1:.4f}")
    print(f"Probabilidad simulada (rvs):    {prob_maximo_1_simulada:.4f}")
    diferencia = abs(prob_maximo_1 - prob_maximo_1_simulada)
    print(f"Diferencia: {diferencia:.4f}")
    assert diferencia < 0.03, "La simulación debería acercarse a la probabilidad teórica"
    print("Ejercicio 5 OK")


# Cuando hayas completado la celda anterior, descomenta la siguiente línea:
# verificar_ejercicio_5()


---
## Ejercicio 6 (difícil) — Pruebas de hipótesis, de principio a fin

Contrasta si la severidad media de los siniestros de `SALUD` es significativamente distinta de la de `VIDA` (t-test de dos muestras independientes, sin asumir varianzas iguales — usad `equal_var=False`). Guarda el resultado completo del test en `resultado_salud_vida` (el objeto que devuelve `scipy.stats.ttest_ind`).

Después, construye la tabla de contingencia entre `producto` y `anulada` sobre `cartera` (¡no `siniestros`, esta vez!) y ejecuta `chi2_contingency`. Guarda el p-valor en `p_valor_chi2_anulacion`.

Finalmente, escribe (en una variable `str`, `conclusion_negocio`) una frase de una línea, en lenguaje de negocio, resumiendo si el producto está asociado con la probabilidad de anulación — coherente con `p_valor_chi2_anulacion` (sin usar la IA para este último paso: es un buen momento para practicar vuestra propia traducción antes de comparar con la del notebook de soluciones).

**Criterio de validación**: `resultado_salud_vida.pvalue` debe ser un `float` entre 0 y 1; la conclusión en `conclusion_negocio` debe ser coherente con si `p_valor_chi2_anulacion < 0.05` o no (si dice "sí hay asociación" cuando el p-valor es alto, o viceversa, está mal, con independencia de lo bien escrita que esté la frase).


In [ ]:
# Tu código aquí
pass


In [ ]:
def verificar_ejercicio_6():
    print(f"p-valor SALUD vs VIDA: {resultado_salud_vida.pvalue:.4f}")
    assert 0 <= resultado_salud_vida.pvalue <= 1

    print(f"p-valor chi2 producto vs anulada: {p_valor_chi2_anulacion:.4f}")
    hay_asociacion = p_valor_chi2_anulacion < ALFA
    print(f"¿Hay asociación estadísticamente significativa? {hay_asociacion}")
    print(f"\nTu conclusión: {conclusion_negocio}")

    if hay_asociacion:
        assert any(
            palabra in conclusion_negocio.lower()
            for palabra in ["sí", "asociación", "asociado", "relacion", "relación"]
        ), "El p-valor indica asociación significativa -- la conclusión debería reflejarlo"
    else:
        assert "no" in conclusion_negocio.lower(), (
            "El p-valor NO indica asociación significativa -- la conclusión debería reflejarlo"
        )
    print("Ejercicio 6 OK")


# Cuando hayas completado la celda anterior, descomenta la siguiente línea:
# verificar_ejercicio_6()


---
## Fin

Si has completado los 6 ejercicios con sus validaciones en verde, has cerrado el CRUD de SQLModel, consumido una API con autenticación, y aplicado distribuciones de probabilidad y pruebas de hipótesis con SciPy sobre datos reales del curso. Compara tu solución con `08_soluciones.ipynb`.
